# Link POI to pre-defined activity purposes

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [23]:
# Load libs
import pandas as pd
import overturemaps
from shapely import wkb
import numpy as np
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
from tqdm import tqdm
import numpy as np
import time
from p_tqdm import p_map
import pickle
from datetime import timedelta
from math import ceil
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.dates as mdates
import h3.api.numpy_int as h3
from shapely.geometry import Polygon
import openai
from openai import OpenAI
import yaml
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
with open('dbs/keys.yaml') as f:
    keys_manager = yaml.load(f, Loader=yaml.FullLoader)
openai.api_key = keys_manager['openai']['key']

## 1. Load POI data

In [4]:
gdf_poi = gpd.read_file('dbs/geo/pois.gpkg')
gdf_poi.head()

,id,source,primary,secondary,confidence,tag,geometry
0,08f39664526034000351dad4f8251283,meta,bed_and_breakfast,"['tours', 'travel']",0.953741,Essential needs,POINT (0.03299 43.10662)
1,08f396645088574d037e9bb4d665fcc3,meta,art_gallery,['sculpture_statue'],0.953741,Social & Leisure,POINT (0.03444 43.14077)
2,08f396645340a52d0352208228386dd4,meta,farm,['fruits_and_vegetables'],0.948077,Other,POINT (0.0515 43.11061)
3,08f39664534598610361c1411eea266d,meta,bar,['farmers_market'],0.954320,Social & Leisure,POINT (0.05538 43.11075)
4,08f396645ed9a4b4038533a3f3bfaeba,meta,real_estate_agent,"['real_estate', 'photography_museum']",0.953741,Other,POINT (0.07098 43.11994)


In [ ]:
def combine_primary_secondary(row):
    if row['secondary'] is None:
        return row['primary']
    else:
        return row['primary'] + ' / ' + ' | '.join(eval(row['secondary']))

In [ ]:
tqdm.pandas(desc="Combining primary and secondary POI types")
gdf_poi.loc[:, 'label'] = gdf_poi.progress_apply(combine_primary_secondary, axis=1)

In [15]:
print(gdf_poi['label'].nunique())
gdf_poi['label'].value_counts().sort_values(ascending=False).head(20)

312676


label
professional_services / professional_services                    23918
beauty_and_spa / beauty_salon                                    13834
beauty_and_spa / beauty_salon | hair_salon                       12574
restaurant / diner                                                6995
real_estate / real_estate_agent                                   6981
beauty_and_spa / beauty_salon | spas                              6686
beauty_salon / beauty_and_spa                                     6544
beauty_and_spa / beauty_salon | barber                            6384
community_services_non_profits / social_service_organizations     5367
retail / retail                                                   4586
insurance_agency / insurance_agency                               4489
real_estate / professional_services                               4362
eyewear_and_optician / optometrist                                4167
shopping / shopping                                               4097


In [22]:
gdf_poi['tag'].unique()

array(['Essential needs', 'Social & Leisure', 'Other',
       'Civic and utility', 'Health services', 'Restaurant', 'Education'],
      dtype=object)

## 2. Get HEALTH, PURCHASE, and LEISURE
LEISURE = "Social & Leisure"

HEALTH = "Health services"

PURCHASE = a subset of Essential needs

In [25]:
purchase_candidates = list(set(gdf_poi.loc[gdf_poi['tag'] == 'Essential needs', 'primary'].to_list()))

In [ ]:
labels_dict = {}
index = 0

out = widgets.Output()

def label_item(label):
    global index
    labels_dict[purchase_candidates[index]] = label
    index += 1
    with out:
        clear_output(wait=True)
        if index < len(purchase_candidates):
            print(f"Item {index+1}/{len(purchase_candidates)}: {purchase_candidates[index]}")
        else:
            print("✅ Labeling complete!")

yes_button = widgets.Button(description="PURCHASE ✅", button_style="success")
no_button = widgets.Button(description="Not PURCHASE ❌", button_style="danger")

yes_button.on_click(lambda b: label_item(1))
no_button.on_click(lambda b: label_item(0))

display(widgets.HBox([yes_button, no_button]), out)

# Initialize first item
with out:
    print(f"Item {index+1}/{len(purchase_candidates)}: {purchase_candidates[index]}")

In [30]:
gdf_poi.head()

,id,source,primary,secondary,confidence,tag,geometry,label
0,08f39664526034000351dad4f8251283,meta,bed_and_breakfast,"['tours', 'travel']",0.953741,Essential needs,POINT (0.03299 43.10662),bed_and_breakfast / tours | travel
1,08f396645088574d037e9bb4d665fcc3,meta,art_gallery,['sculpture_statue'],0.953741,Social & Leisure,POINT (0.03444 43.14077),art_gallery / sculpture_statue
2,08f396645340a52d0352208228386dd4,meta,farm,['fruits_and_vegetables'],0.948077,Other,POINT (0.0515 43.11061),farm / fruits_and_vegetables
3,08f39664534598610361c1411eea266d,meta,bar,['farmers_market'],0.954320,Social & Leisure,POINT (0.05538 43.11075),bar / farmers_market
4,08f396645ed9a4b4038533a3f3bfaeba,meta,real_estate_agent,"['real_estate', 'photography_museum']",0.953741,Other,POINT (0.07098 43.11994),real_estate_agent / real_estate | photography_...


In [31]:
def poi2purpose(row):
    if row['tag'] == 'Essential needs':
        if row['primary'] in labels_dict:
            return 'PURCHASE' if labels_dict[row['primary']] == 1 else 'OTHER'
        else:
            return 'OTHER'
    elif row['tag'] == 'Health services':
        return 'HEALTH'
    elif row['tag'] == 'Social & Leisure':
        return 'LEISURE'
    else:
        return 'OTHER'
    
tqdm.pandas(desc="Assigning purposes to POIs")
gdf_poi.loc[:, 'purpose'] = gdf_poi.progress_apply(poi2purpose, axis=1)

Assigning purposes to POIs: 100%|██████████| 2098361/2098361 [00:30<00:00, 69057.06it/s]


In [32]:
gdf_poi['purpose'].value_counts()

purpose
OTHER       1081791
LEISURE      567513
PURCHASE     262817
HEALTH       186240
Name: count, dtype: int64

In [33]:
gdf_poi.to_file('dbs/geo/pois_p.gpkg', driver='GPKG', layer='pois')